In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from rouge_score import rouge_scorer
from sklearn.model_selection import train_test_split
import numpy as np
import re
import emoji
import time
import pandas as pd


In [ ]:
# Сбор данных. Скачать датасет с короткими постами

tweets = []
with open('data/tweets.txt', 'r', encoding='utf-8') as f:
    for line in f:
        text = line.strip()
        if text:  # пропускаем пустые строки
            tweets.append(text)

df = pd.DataFrame(tweets, columns=['text'])
print(df.head())
#сырой датасет
df.to_csv('data/raw_dataset.csv', index=False, encoding='utf-8')

In [ ]:
# Очистка и нормализация текста

from data_utils import preprocess_tweet

# Применяем функцию к колонке 'text'
df['tokens'] = df['text'].apply(preprocess_tweet)

# Посмотреть результат
print(df[['text', 'tokens']].head())

In [ ]:
#очищенный датасет
df.to_csv('data/dataset_processed.csv', index=False, encoding='utf-8')

In [ ]:
# Разделение на трейн, валидацию и тест
# Исходные данные: список списков токенов
from sklearn.model_selection import train_test_split
from traintestval import create_training_sequences


all_tweets = df['tokens'].tolist()

# Шаг 1: Отделяем тест (10% от всего датасета)
train_tweets, test_tweets = train_test_split(
    all_tweets, 
    test_size=0.1, 
    random_state=42,  # Фиксируем для воспроизводимости результатов
    shuffle=True      # Перемешиваем перед разделением
)

# Шаг 2: От оставшихся 90% отделяем валидацию 
train_tweets, val_tweets = train_test_split(
    train_tweets, 
    test_size=0.1 / 0.9,  # Коэффициент для получения 10% от общего объема
    random_state=42,
    shuffle=True
)
# Шаг 3: # Сохранение - трейн, валидация и тест
pd.DataFrame({'tokens': train_tweets}).to_csv(
    'data/train.csv', 
    index=False, 
    encoding='utf-8')
pd.DataFrame({'tokens': test_tweets}).to_csv(
    'data/test.csv', 
    index=False, 
    encoding='utf-8')
pd.DataFrame({'tokens': val_tweets}).to_csv(
    'data/val.csv', 
    index=False, 
    encoding='utf-8')



# Шаг 4: # Формирование обучающих примеров
seq_len = 7  # Длина контекста
# Теперь создаём пары (X, y) независимо для каждой части
X_train, y_train = create_training_sequences(train_tweets, seq_len=seq_len)
X_val, y_val = create_training_sequences(val_tweets, seq_len=seq_len)
X_test, y_test = create_training_sequences(test_tweets, seq_len=seq_len)


In [ ]:
#создание словаря
from collections import Counter

def build_vocab(tokens_list, min_freq=2):
    counter = Counter(token for tokens in tokens_list for token in tokens)
    special_tokens = ['<pad>', '<unk>', '<eos>']
    vocab_list = special_tokens + [t for t, c in counter.items() if c >= min_freq]
    return {t: i for i, t in enumerate(vocab_list)}, {i: t for t, i in enumerate(vocab_list)}

# Строим словарь на train
token2idx, idx2token = build_vocab(train_tweets, min_freq=2)
vocab_size = len(token2idx)
print(f"Размер словаря: {vocab_size}")

In [ ]:
# dataloader
from next_token_dataset import create_dataloader

BATCH_SIZE = 256

train_loader = create_dataloader(X_train, y_train, token2idx, batch_size=256, shuffle=True)
val_loader =  create_dataloader(X_val, y_val, token2idx, batch_size=256, shuffle=True)
test_loader =  create_dataloader(X_test, y_test, token2idx, batch_size=256, shuffle=True)

In [ ]:
# Импортируем модель
from lstm_model import LSTMAutoCompleter  # Импортируем модель

# Создание модели
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTMAutoCompleter(
    vocab_size=vocab_size,
    embed_dim=128,
    hidden_dim=256,
    num_layers=2,
    dropout=0.3
).to(device)

In [ ]:
from lstm_train import train_model, load_model

# === 3. ОБУЧЕНИЕ ===
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    token2idx=token2idx,
    idx2token=idx2token,
    epochs=10,
    lr=0.001,
    device=device,
    save_path='best_lstm_model.pth'
)

# === 4. ЗАГРУЗКА ЛУЧШЕЙ МОДЕЛИ ===
model = load_model(model, 'best_lstm_model.pth', device)

In [ ]:
# оценка
from eval_lstm import evaluate_rouge, print_evaluation_results, save_results

metrics = evaluate_rouge(
    model=model,
    data_loader=test_loader,
    token2idx=token2idx,
    idx2token=idx2token,
    device=device,
    max_examples=500,
    max_gen_len=15,
    temperature=0.8
)

# === 4. ВЫВОД РЕЗУЛЬТАТОВ ===
print_evaluation_results(metrics, model_name='LSTM')

# === 5. СОХРАНЕНИЕ ===
save_results(metrics, filepath='results/lstm_metrics.csv')

In [ ]:
# трансформер
from eval_transformer_pipeline import (
    load_transformer_model,
    tune_generation_params,
    evaluate_rouge_transformer,
    print_transformer_results,
    save_transformer_results,
    compare_lstm_vs_transformer
)

# === 1. ЗАГРУЗКА МОДЕЛИ ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, tokenizer = load_transformer_model('distilgpt2', device)

# === 2. ПОДБОР ПАРАМЕТРОВ (на валидации) ===
param_grid = [
    {'max_length': 30, 'temperature': 0.7, 'top_k': 50, 'top_p': 0.9},
    {'max_length': 30, 'temperature': 0.8, 'top_k': 50, 'top_p': 0.9},
    {'max_length': 30, 'temperature': 1.0, 'top_k': 50, 'top_p': 0.9},
    {'max_length': 40, 'temperature': 0.8, 'top_k': 50, 'top_p': 0.9},
]
best_config, _ = tune_generation_params(model, tokenizer, val_pairs, param_grid, device)

# === 3. ФИНАЛЬНАЯ ОЦЕНКА (на тесте) ===
test_metrics = evaluate_rouge_transformer(
    model, tokenizer, test_pairs, best_config, device, max_examples=500
)

# === 4. ВЫВОД И СОХРАНЕНИЕ ===
print_transformer_results(test_metrics)
save_transformer_results(test_metrics, best_config)


In [ ]:
# Загружаем метрики из CSV
lstm_df = pd.read_csv('results/lstm_metrics.csv')

# Преобразуем в формат словаря (как возвращает evaluate_rouge)
lstm_metrics = {
    'rouge1_mean': lstm_df[lstm_df['metric'] == 'rouge1_mean']['value'].values[0],
    'rouge1_std': lstm_df[lstm_df['metric'] == 'rouge1_std']['value'].values[0],
    'rouge2_mean': lstm_df[lstm_df['metric'] == 'rouge2_mean']['value'].values[0],
    'rouge2_std': lstm_df[lstm_df['metric'] == 'rouge2_std']['value'].values[0],
    'n_samples': lstm_df[lstm_df['metric'] == 'n_samples']['value'].values[0],
    'examples': []  # Примеры можно не загружать для сравнения
}

In [ ]:
compare_lstm_vs_transformer(lstm_metrics, trans_metrics)